In [0]:
from pyspark.sql import SparkSession

spark=SparkSession.builder.appName("Spark DataFrame").getOrCreate()


In [0]:
df1=spark.read.format("csv").option("header",True).load("/Volumes/workspace/default/assigment5/asigment5.csv")x`x`

df1.printSchema()

In [0]:
df=spark.read.format("csv").option("inferSchema",True).option("header",True).load("/Volumes/workspace/default/assigment5/asigment5.csv")

#header - tells whether first line of csv file contains column names or not
#inferSchema - tells whethere to automatically detect and assign data types to each column 

df.show(10)

In [0]:
df.printSchema()

### Read Modes

In [0]:
# permissive - default behavior (parse the records and handles malformed data)
#drop malformed - malformed records are discarded
#failfast - if spark encounters malformed data the read fails


### Delimiter

In [0]:
# defines the seprator 
# for a pipe seprated file ex 1|rahul|IT|7000
# here the delimiter is (|) use .option("delimiter","|")

### Explicit Schema and inferSchema

In [0]:
'''
inferSchema-
    spark determines the type 
    can behave unexpectedly with messy data 
    may broke in productions 
    usage: .options("inferSchema",True)

Explicit Schema -
    define the schema manually 
    more controlled
    use two keys as StructType() and StructField()
    usage : 
        schema=StructType([
                StructField("column",data_type,nullable)
            ])
'''

|-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- price: double (nullable = true)
 |-- subscription: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- store_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- raw_timestamp: string (nullable = true)

In [0]:
from pyspark.sql.types import *
schema=StructType([
    StructField("user_id",IntegerType(),True),
    StructField("Transaction_date",DateType(),True),
    StructField("city",StringType(),True),
    StructField("product_category",StringType(),True),
    StructField("sale_amount",DoubleType(),True),
    StructField("price",DoubleType(),True),
    StructField("subscription",StringType(),True),
    StructField("age",IntegerType(),True),
    StructField("store_id",StringType(),True),
    StructField("status",StringType(),True),
    StructField("email",StringType(),True),
    StructField("username",StringType(),True),
    StructField("raw_timestamp",StringType(),True)
])

In [0]:
df2=spark.read.format("csv").schema(schema).option("header",True).load("/Volumes/workspace/default/assigment5/asigment5.csv")

df2.printSchema()

In [0]:
df3=spark.read.format("csv").option("header",True).option("mode","PERMISSIVE").load("/Volumes/workspace/default/pratice/file.csv")

df3.show()

In [0]:
df4=spark.read.format("csv").option("header",True).option("mode","dropMalformed").load("/Volumes/workspace/default/pratice/file.csv")

df4.show()

In [0]:
df5=spark.read.format("csv").option("header",True).option("mode","failFast").load("/Volumes/workspace/default/pratice/file.csv")

df5.show()

### Parquet and ORC 

In [0]:
'''
csv- row oriented


    row 1 -> id , name , department , salary
    row 2 -> id , name , dpeartment , salary

parquet- column oriented


id:   name:   department:   salary:
1     ravi      IT         70000
2     rahul     HR         50000
3     priya     IT         80000

if i need only salary column spark can avoid reading unnecessay columns 
this is known as column pruning

'''

In [0]:
from pyspark.sql import Row

data = [
    Row(id=1, name="Ravi", department="IT", salary=70000),
    Row(id=2, name="Rahul", department="HR", salary=50000),
    Row(id=3, name="Priya", department="IT", salary=80000),
    Row(id=4, name="Anita", department="Finance", salary=65000),
    Row(id=5, name="Kiran", department="HR", salary=55000),
    Row(id=6, name="Sneha", department="IT", salary=90000)
]

df = spark.createDataFrame(data)

df.show()
df.printSchema()

In [0]:
df.write.parquet("/Volumes/workspace/default/pratice/employee_parquet")

In [0]:
parquet_df=spark.read.parquet("/Volumes/workspace/default/pratice/employee_parquet")
parquet_df.show()

parquet_df.printSchema()

'''
we dont provide header=True or inferSchemA=True
becase parquet stores schema information along with data
'''

### Write modes

In [0]:
'''
write modes
append -Adds the new data alongside existing files without deleting anything
overwrite - Deletes or replaces existing files at the destination and writes the new data.
ignore -Silently does nothing (no write operation occurs, no data changes, no error thrown).
error - Throws an AnalysisException (PATH_ALREADY_EXISTS) and halts execution.
'''

In [0]:
df.write.mode("overwrite").parquet("/Volumes/workspace/default/pratice/employee_parquet")

In [0]:
df.write.mode("append").parquet("/Volumes/workspace/default/pratice/employee_parquet")

In [0]:
df.write.mode("ignore").parquet("/Volumes/workspace/default/pratice/employee_parquet")

In [0]:
df.write.mode("error").parquet("/Volumes/workspace/default/pratice/employee_parquet")

In [0]:
df5=spark.read.parquet("/Volumes/workspace/default/pratice/employee_parquet")
df5.select("id","name").show()

In [0]:
df5.select("salary").show()

# performs column pruning where it doest need to process every column

### Recursive File lookup

In [0]:
'''
imagine this directory

data/
├── 2024/
│   ├── employees1.parquet
│   └── employees2.parquet
│
├── 2025/
│   ├── employees3.parquet
│   └── employees4.parquet
│
└── 2026/
    ├── employees5.parquet
    └── employees6.parquet
normally
'''

spark.read.parquet("data/*")
# may not automatically recursively search arbitary nested directories

#to search recursively use .option("recursiveFileLookup", "true")
spark.read.option("recursiveFileLookup", "true").parquet("data/")

In [0]:
df.write.mode("overwrite").parquet("/Volumes/workspace/default/pratice/employee/2025")
df.write.mode("overwrite").parquet("/Volumes/workspace/default/pratice/employee/2026")

In [0]:
df = spark.read.parquet("/Volumes/workspace/default/pratice/employees")

In [0]:
df = spark.read \
    .option("recursiveFileLookup", "true") \
    .parquet("/Volumes/workspace/default/pratice/employees")

In [0]:
df_json=spark.read.json("/Volumes/workspace/default/pratice/emp.json")

In [0]:
df_json.show()

In [0]:
df_json.printSchema()

In [0]:
# for a nested json the data is stored in a struct column 

In [0]:
df_nested_json=spark.read.json("/Volumes/workspace/default/pratice/empas.json")

In [0]:
df_nested_json.show()

In [0]:
df_nested_json.printSchema() #schema is struct not string 

In [0]:
# use . to access the nested column in data frame 

df_nested_json.select(
    "id",
    "name",
    "address.city",
    "address.state"
).show()

In [0]:
#json arrays are stored as of arrays of struct 
# if we want to access each and every element of the array and display we need to use explode()

In [0]:
df_json_arr = spark.read.option("multiline", "true").json("/Volumes/workspace/default/pratice/emparr.json")
#enable multiline option to read an array 
df_json_arr.show()
df_json_arr.printSchema()

In [0]:
df_json_arr.select("name", "skills").show(truncate=False)

In [0]:
from pyspark.sql.functions import explode

skills_df = df_json_arr.select(
    "id",
    "name",
    explode("skills").alias("skill")
)

skills_df.show()

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Sample DataFrame with raw JSON in a string column
data = [(1, '{"city": "Hyderabad", "state": "Telangana", "pincode": 500081}')]
df = spark.createDataFrame(data, ["id", "address_raw"])
df.show()

# 1. Define schema for the JSON content
schema = "city STRING, state STRING, pincode INT"
# Or: StructType([StructField("city", StringType()), StructField("state", StringType()), StructField("pincode", IntegerType())])

# 2. Parse the string into a struct column
parsed_df = df.withColumn("address", from_json(col("address_raw"), schema))

# Now you can access nested fields directly using dot notation:
parsed_df.select("id", "address.city", "address.state", "address.pincode").show()

In [0]:
from pyspark.sql.functions import to_json, col

# Pack the structured 'address' back into a single string column
json_df = parsed_df.select("id", to_json(col("address")).alias("address_json_str"))

json_df.show(truncate=False)

In [0]:
data2=[
    (1, "Ravi", 50000),
    (2, "Rahul", 65000),
    (3, "Priya", 75000),
    (4, "Anita", 90000),
    (5, "Kiran", 45000)
]

columns=["id","name","salary"]

df9 = spark.createDataFrame(data2, columns)

df9.show()

In [0]:
def salary_category(salary):
    if salary>=80000:
        return "High"
    elif salary>=60000:
        return "Medium"
    else:
        return "Low"
    

In [0]:
salary_udf=udf(salary_category,StringType())


In [0]:
res=df9.withColumn(
    "salary_category",
    salary_udf("salary")
)

res.show()

In [0]:
data9 = [
    (1, "Rahul", 50000, 23),
    (2, "Priya", 75000, 27),
    (3, "Arjun", 90000, 31),
    (4, "Sneha", 45000, 22),
    (5, "Kiran", 65000, 26)
]


df = spark.createDataFrame(
    data9,
    ["id", "name", "salary", "age"]
)

df.show()

In [0]:
from pyspark.sql.types import StringType
def categorize_salary(salary):
    if salary>=70000:
        return "High"
    elif salary<70000:
        return "Normal"

salary_category_df=udf(categorize_salary,StringType())

res=df.withColumn(
    "salary_category",
    salary_category_df("salary")
)

res.show()

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import col

def category_age(age):
    if age is None:
        return None
    if age<25:
        return "Young"
    elif 25<=age<30:
        return "Adult"
    else:
        return "Senior"
    
age_category_df=udf(category_age,StringType())

res=res.withColumn(
    "age_category",
   age_category_df(col("age"))
)

res.show()

In [0]:
from pyspark.sql.types import DoubleType
def salary_increase(salary):
    if salary is None:
        return None
    if salary:
        return salary+(0.10*salary)
    
salary_increase_df=udf(salary_increase,DoubleType())

res=res.withColumn(
    "salary_increment",
    salary_increase_df("salary")
)

res.show()

In [0]:
def multiple_input(salary,age):
    if salary is None or age is None:
        return None
    if salary>=80000 and age>=30:
        return "A"
    elif salary>=60000 :
        return "B"
    else:
        return "C"
    
multiple_input_df=udf(multiple_input,StringType())

res=df.withColumn(
    "multiple_input",
    multiple_input_df("salary","age")
)
res.show()

    

In [0]:
data10 = [
    (1, "Rahul", "+91-9876543210"),
    (2, "Priya", "98765 43210"),
    (3, "Arjun", "+91 9988776655"),
    (4, "Sneha", None),
    (5, "Kiran", "91-9123456789")
]

df = spark.createDataFrame(
    data10,
    ["customer_id", "name", "phone"]
)

df.show()


In [0]:
df.printSchema()

Scenario-1

In [0]:
def standardize_phone_numbers(phone):
    if phone is None:
        return None
    s=""
    for i in phone:
        if i.isdigit():
            s+=i
    if len(s)>10:
        s=s[2:len(s)]
    return s

standardize_phone_numbers_df=udf(standardize_phone_numbers,StringType())

res=df.withColumn(
    "standardized_phone",
    standardize_phone_numbers_df("phone")
)

res.show()


Scenario-2

In [0]:
res.show()

In [0]:
def classify_phone(phone):
    if phone is None:
        return None
    if len(phone)==10:
        return "Valid"
    else:
        return "Invalid"
    
classify_phone_df=udf(classify_phone,StringType())

ans=res.withColumn(
    "classify_phone_number",
    classify_phone_df("standardized_phone")
)

ans.show()